In [ ]:
%pip install nextrec --quiet --upgrade

# 5-Minute Quick Start

This notebook introduces NextRec, a unified, efficient, and scalable recommender-system framework, and walks you through training and building a production-ready model from scratch. The example uses internal feature definitions and online samples from E-commerce scenario.

Before getting started, install nextrec from the command line:

```bash
# Release
pip install nextrec

# Test
pip install -i https://test.pypi.org/simple/ nextrec
```

Here is a quick primer on the signals we usually process in recommendation. We handle several input types, transform them, and then feed vectors into the network:

- Dense features (numeric): continuous or ordered values such as age, price, duration, or scores; typically standardized/normalized or log-transformed.
- Sparse features (categorical/ID): high-cardinality discrete fields such as user ID, item ID, gender, occupation, or device type; typically indexed and embedded via an embedding lookup matrix.
- Sequence features (behavior history): variable-length histories such as browse/click/purchase lists. They capture user behavior and interest drift; we usually truncate/pad, embed, and then aggregate (mean/sum/attention) to get a fixed-length vector.
- Context features: environment information such as time, geography, or slot position; can be dense or sparse and often interacts with the main features.
- Multi-modal features: vectors from pre-trained models on text, images, or video; they can be used directly as dense inputs or interact with IDs.

A typical training data format looks like this:

```text
user_id,item_id,gender,age,occupation,history_seq,label
1024,501,1,28,3,"[12,45,18,77]",1
2048,777,0,35,5,"[8,99]",0
```

We provide a desensitized e-commerce dataset with user IDs, item IDs, dense features, sparse features, and sequence features. The labels include both click and conversion.


In [ ]:
pip show nextrec

In [ ]:
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/zerolovesea/NextRec/main/dataset/multitask_task.csv')
df.head()

In [ ]:
task_labels = ['click', 'conversion']
dense_features_list = [col for col in df.columns if 'dense' in col]
sparse_features_list = [col for col in df.columns if 'sparse' in col] + ['user_id', 'item_id']
sequence_features_list = [col for col in df.columns if 'sequence' in col]

Next we prepare the model by defining the different feature types it needs and passing them into the model. Here we use the built-in DenseFeature, SequenceFeature, and SparseFeature classes from nextrec.

In [ ]:
from nextrec.basic.features import DenseFeature, SequenceFeature, SparseFeature

# we treat all dense features as DenseFeature, proj_dim=1 means no projection is performed. When proj_dim is greater than 1, 
# it indicates that a linear transformation is performed on the dense features, similar to the effect of embedding
dense_features = [DenseFeature(name=feat, proj_dim=1) for feat in dense_features_list] 

# Sparse features and sequence features are generally embedded, and the embedding_dim can be adjusted according to actual needs
sparse_features = []
for feat in sparse_features_list:
    vocab_size = 20001 # assuming the vocabulary size for each sparse feature is 20001
    # SparseFeature can also set some other parameters, such as initializer, regularization, and embedding_name, etc. 
    # When two features share embedding, the same embedding_name can be set       
    sparse_features.append(SparseFeature(name=feat, vocab_size=vocab_size, embedding_dim=4, embedding_name=feat)) 

# Sequence features are handled similarly to sparse features, but you also need to set the maximum length max_len and padding_idx parameters
sequence_features = []
for feat in sequence_features_list:
    vocab_size = 500 # assuming the vocabulary size for each sequence feature is 500
    sequence_features.append(
        SequenceFeature(
            name=feat,
            vocab_size=vocab_size,
            max_len=20,
            embedding_dim=8,
            padding_idx=0
        )
    )

After defining the features, let's choose a model to train. NextRec offers more than 20 industry-standard models for retrieval, ranking, and multi-task learning. Here we start with a classic MMOE model.

Before training, we instantiate the model and configure parameters, optimizer, scheduler, and loss function. NextRec supports more than 8 optimizers, 10 schedulers, 20 loss functions, and imbalance-aware losses.


In [ ]:
from nextrec.models.multitask.mmoe import MMOE

# For MMOE, configure expert networks and task towers. Here we use 4 experts with two layers each,
# and two task towers with two layers each. We have two binary tasks: click and conversion.
model = MMOE(
    dense_features=dense_features,
    sparse_features=sparse_features,
    sequence_features=sequence_features,
    expert_mlp_params={"hidden_dims": [128, 64], "activation": "leaky_relu", "dropout": 0.3},
    num_experts=4,
    tower_mlp_params_list=[
        {"hidden_dims": [64, 32], "activation": "leaky_relu", "dropout": 0.2},
        {"hidden_dims": [64, 32], "activation": "leaky_relu", "dropout": 0.2},
    ],
    target=task_labels,
    task=['binary', 'binary'],
    device='cpu',
    embedding_l1_reg=1e-6,
    embedding_l2_reg=1e-5,
    dense_l1_reg=1e-5,
    dense_l2_reg=1e-4,
    session_id="mmoe_task"
)

# Compile model: optimizer and loss are configured in compile().
model.compile(
    optimizer="adam",
    optimizer_params={"lr": 1e-3, "weight_decay": 1e-5},
    loss=['bce', 'bce'],
)

# Train for 1 epoch and set per-task metrics.
model.fit(
    train_data=df,
    metrics={
        'click': ['auc', 'recall', 'precision'],
        'conversion': ['auc', 'recall', 'precision'],
    },
    epochs=1,
    valid_split=0.2,
)


You can control train/validation split via `fit` parameters `train_data`, `valid_data`, and `valid_split`. `train_data` and `valid_data` support DataFrame, dict, and file paths.

- If `valid_data` is provided, it is used as the validation set.
- If `train_data` and `valid_split` are provided, validation data is split from training data by ratio.


In [ ]:
from sklearn.model_selection import train_test_split

# Split training and validation data
train_df, valid_df = train_test_split(df, test_size=0.2, random_state=2025)

model.fit(
    train_data=train_df,
    valid_data=valid_df,
    metrics={
        'click': ['auc', 'recall', 'precision'],
        'conversion': ['auc', 'recall', 'precision'],
    },
    epochs=1,
)

Next we train a ranking model using AutoINT as the example, switching the task from multi-task to single-task. This model comes from a Peking University paper published at CIKM 2019; you can read an explainer [here](https://guyuecanhui.github.io/2020/05/09/paper-2019-pku-autoint/).


In [ ]:
from nextrec.models.ranking.autoint import AutoInt

target = 'conversion'

model = AutoInt(
    dense_features=dense_features,
    sparse_features=sparse_features,
    sequence_features=sequence_features,
    att_layer_num=3,
    att_embedding_dim=8,
    att_head_num=2,
    att_dropout=0.0,
    att_use_residual=True,
    target=target,
    device='cpu',
    embedding_l1_reg=1e-6,
    dense_l1_reg=1e-5,
    embedding_l2_reg=1e-5,
    dense_l2_reg=1e-4,
    session_id="autoint_task"
)

# Compile model
model.compile(
    optimizer="adam",
    optimizer_params={
        "lr": 1e-3,
        "weight_decay": 1e-5,
    },
    loss="bce",
)

# Train model
model.fit(
    train_data=df,
    valid_split=0.2,
    metrics=['auc', 'recall', 'precision'],
    epochs=1,
    batch_size=512,
    shuffle=True,
)

Below are the models currently supported—feel free to try them out.

### Ranking models

| Model | Paper | Year | Status |
|------|------|------|------|
| **FM** | Factorization Machines | ICDM 2010 | Supported |
| **AFM** | Attentional Factorization Machines: Learning the Weight of Feature Interactions via Attention Networks | IJCAI 2017 | Supported |
| **DeepFM** | DeepFM: A Factorization-Machine based Neural Network for CTR Prediction | IJCAI 2017 | Supported |
| **Wide&Deep** | Wide & Deep Learning for Recommender Systems | DLRS 2016 | Supported |
| **xDeepFM** | xDeepFM: Combining Explicit and Implicit Feature Interactions | KDD 2018 | Supported |
| **FiBiNET** | FiBiNET: Combining Feature Importance and Bilinear Feature Interaction for CTR Prediction | RecSys 2019 | Supported |
| **PNN** | Product-based Neural Networks for User Response Prediction | ICDM 2016 | Supported |
| **AutoInt** | AutoInt: Automatic Feature Interaction Learning | CIKM 2019 | Supported |
| **DCN** | Deep & Cross Network for Ad Click Predictions | ADKDD 2017 | Supported |
| **DIN** | Deep Interest Network for Click-Through Rate Prediction | KDD 2018 | Supported |
| **DIEN** | Deep Interest Evolution Network for Click-Through Rate Prediction | AAAI 2019 | Supported |
| **MaskNet** | MaskNet: Introducing Feature-wise Gating Blocks for High-dimensional Sparse Recommendation Data | 2020 | Supported |

### Retrieval models

| Model | Paper | Year | Status |
|------|------|------|------|
| **DSSM** | Learning Deep Structured Semantic Models | CIKM 2013 | Supported |
| **DSSM v2** | DSSM with pairwise BPR-style optimization | - | Supported |
| **YouTube DNN** | Deep Neural Networks for YouTube Recommendations | RecSys 2016 | Supported |
| **MIND** | Multi-Interest Network with Dynamic Routing | CIKM 2019 | Supported |
| **SDM** | Sequential Deep Matching Model | - | Supported |

### Multi-task models

| Model | Paper | Year | Status |
|------|------|------|------|
| **MMOE** | Modeling Task Relationships in Multi-task Learning | KDD 2018 | Supported |
| **PLE** | Progressive Layered Extraction | RecSys 2020 | Supported |
| **ESMM** | Entire Space Multi-Task Model | SIGIR 2018 | Supported |
| **ShareBottom** | Multitask Learning | - | Supported |
